In [1]:
import json
import random

try:
    from num2words import num2words
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "num2words"])
    from num2words import num2words

num_samples = 2500
output_file = "semantic_redundancy_train_json.json"

def bsc_channel(text, ber):
    """
    模擬二元對稱通道 (BSC) 雜訊干擾
    """
    binary_data = ''.join(format(ord(char), '08b') for char in text)
    noisy_binary_list = []

    for bit in binary_data:
        if random.random() < ber:
            noisy_binary_list.append('1' if bit == '0' else '0')
        else:
            noisy_binary_list.append(bit)

    noisy_binary_str = ''.join(noisy_binary_list)

    chars = []
    for i in range(0, len(noisy_binary_str), 8):
        byte = noisy_binary_str[i:i+8]
        if len(byte) == 8:
            try:
                chars.append(chr(int(byte, 2)))
            except Exception:
                chars.append('?')
    return ''.join(chars)

# =====================================================================
# 核心升級：導入全方位工業 IoT 知識圖譜 (Ontology)
# 包含電氣、流體、機械、熱力學四大領域，並定義了合理的物理數值範圍
# =====================================================================
comprehensive_ontology = {
    "Electrical": {
        "Main_Grid_Voltage_V":    {"range": (110.0, 480.0),  "type": "float"},
        "Battery_Voltage_mV":     {"range": (3000, 4200),    "type": "int"},
        "Motor_Current_A":        {"range": (0.5, 100.0),    "type": "float"},
        "Phase_Frequency_Hz":     {"range": (49.5, 60.5),    "type": "float"}
    },
    "Fluid": {
        "Hydraulic_Pressure_bar": {"range": (50.0, 350.0),   "type": "float"},
        "Coolant_Flow_Rate_LPM":  {"range": (5.0, 200.0),    "type": "float"},
        "Tank_Level_Percent":     {"range": (0.0, 100.0),    "type": "float"}
    },
    "Mechanical": {
        "Engine_rpm":             {"range": (500, 3000),     "type": "int"},
        "Turbine_Speed_rpm":      {"range": (3000, 25000),   "type": "int"},
        "Bearing_Vibration_mmps": {"range": (0.1, 15.0),     "type": "float"},
        "Drive_Shaft_Torque_Nm":  {"range": (50.0, 2000.0),  "type": "float"}
    },
    "Thermal_Env": {
        "Coolant_temp":           {"range": (60, 110),       "type": "int"},
        "Exhaust_Gas_Temp_C":     {"range": (200, 850),      "type": "int"},
        "Furnace_Temp_K":         {"range": (1000, 1500),    "type": "int"},
        "Gas_Concentration_ppm":  {"range": (0, 10000),      "type": "int"}
    }
}

# 將圖譜攤平，方便隨機抽取
flat_sensors = []
for category, sensors in comprehensive_ontology.items():
    for sensor_name, props in sensors.items():
        flat_sensors.append((sensor_name, props))

dataset = []
print(f"正在生成 {num_samples} 筆具備「物理常識」的泛化特訓資料 (Dynamic BER: 0.01 ~ 0.06)...")

for _ in range(num_samples):
    # 1. 從知識圖譜中隨機抽取感測器
    sensor, props = random.choice(flat_sensors)

    # 2. 根據感測器的物理屬性 (type 與 range) 生成合理的 Ground Truth 數值
    min_val, max_val = props["range"]
    if props["type"] == "float":
        val = round(random.uniform(min_val, max_val), 2)
    else:
        val = random.randint(int(min_val), int(max_val))

    val_text = num2words(val).replace("-", " ")

    # 3. 題目 (Input): 帶有數字與英文冗餘對照，並通過 BSC 雜訊通道
    raw_input_text = f"ERR: {sensor} {val} ({val_text})"
    current_ber = random.uniform(0.01, 0.06)
    noisy_input = bsc_channel(raw_input_text, current_ber)

    # 4. 答案 (Output): 依靠語意冗餘校正後的標準 JSON 結構
    json_output = {
        "status": "ERR",
        "sensor": sensor,
        "value": val
    }
    json_output_str = json.dumps(json_output, ensure_ascii=False)

    # 5. 提示詞設計：不再限制感測器名稱，而是強調「利用工業常識與冗餘」
    instruction = "You are a 6G Semantic Decoder in an Industrial IoT environment. Decode the high-noise log by leveraging your general industrial knowledge and semantic redundancy (text in parentheses) to reconstruct the exact sensor name and numeric value. Output the clean result directly as a JSON object."

    dataset.append({
        "instruction": instruction,
        "input": noisy_input,
        "output": json_output_str
    })

# 批次寫入檔案
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

print("=" * 50)
print(f"泛化語意訓練集已成功建立: {output_file}")
print(f"總筆數: {len(dataset)} (涵蓋電氣、流體、機械、熱力學四大領域)")
print("\n[範例預覽]:")
print(f"Input : {dataset[-1]['input']}")
print(f"Output: {dataset[-1]['output']}")
print("=" * 50)

正在生成 2500 筆具備「物理常識」的泛化特訓資料 (Dynamic BER: 0.01 ~ 0.06)...
泛化語意訓練集已成功建立: semantic_redundancy_train_json.json
總筆數: 2500 (涵蓋電氣、流體、機械、熱力學四大領域)

[範例預覽]:
Input : ÅRR:`Cool`nt_Flov_Rate_LPM!1w/48 (seventeen point fouz eight)
Output: {"status": "ERR", "sensor": "Coolant_Flow_Rate_LPM", "value": 17.48}
